<a href="https://colab.research.google.com/github/natchanant-arch/Project_Savings_Cooperative/blob/First/FINAL_PRO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 ธุรกิจสหกรณ์ออมทรัพย์ (Savings Cooperative)

ระบบจำลองบัญชีสหกรณ์ออมทรัพย์ ครอบคลุมการ **ฝาก – ถอน – โอน** โดยสมาชิกทำธุรกรรมได้ทีละรายการ พร้อมตรวจสอบยอดเงินคงเหลือให้เพียงพอก่อนทำรายการทุกครั้ง และคำนวณดอกเบี้ยจากยอดเงินคงเหลือในบัญชีเมื่อสิ้นปี

---

## 📑 สารบัญ

| ส่วน | หัวข้อ |
|:---:|---|
| 0 | Import เพื่อเรียกใช้งานชุดคำสั่ง / ฟังก์ชันสำเร็จรูป |
| 1 | เตรียม Class และฟังก์ชัน |
| 2 | ทดสอบฟังก์ชันทีละตัว ก่อนประกอบเป็นกระบวนการ |
| 3 | ฟังก์ชันอธิบายขั้นตอนคำนวณรายการ |
| 4 | จำลอง "ลูกค้า 1 คนเดินเข้าธนาคาร" แบบ step-by-step |
| 5 | จำลองลูกค้าหลายคนเดินเข้าธนาคารต่อเนื่องกัน |
| 6 | สรุปผล — ยืนยันว่าฟังก์ชันคืนค่าถูกต้องและใช้ต่อได้จริง |
| 7 | ตารางลูกค้า |
| 8 | ตารางธุรกรรม |
| 9 | สรุปผลรวม 300 รายการ |

---

## — Import เพื่อ เรียกใช้งานชุดคำสั่ง หรือฟังก์ชันสำเร็จรูป —

In [1]:
import random
import time
from datetime import datetime, timedelta
!pip install Faker
from faker import Faker
fake = Faker("th_TH")
random.seed(1)
import pandas as pd
import matplotlib, os, shutil
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.7 MB/s eta 0:00:00


In [2]:
# 1. ติดตั้งฟอนต์ภาษาไทย
!apt-get -y install fonts-thai-tlwg

# 2. ล้าง cache ของ matplotlib เพื่ออัปเดตฟอนต์ใหม่
cache_dir = matplotlib.get_cachedir()
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

# 3. ลงทะเบียนฟอนต์ใหม่เข้ากับ FontManager
font_path = '/usr/share/fonts/truetype/tlwg/Loma.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)

# 4. ตั้งค่าฟอนต์หลัก
plt.rcParams['font.family'] = 'Loma'
plt.rcParams['axes.unicode_minus'] = False

print("ตั้งค่าระบบฟอนต์ภาษาไทยสำเร็จ!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-tlwg-garuda fonts-tlwg-garuda-ttf fonts-tlwg-kinnari
  fonts-tlwg-kinnari-ttf fonts-tlwg-laksaman fonts-tlwg-laksaman-ttf
  fonts-tlwg-loma fonts-tlwg-loma-ttf fonts-tlwg-mono fonts-tlwg-mono-ttf
  fonts-tlwg-norasi fonts-tlwg-norasi-ttf fonts-tlwg-purisa
  fonts-tlwg-purisa-ttf fonts-tlwg-sawasdee fonts-tlwg-sawasdee-ttf
  fonts-tlwg-typewriter fonts-tlwg-typewriter-ttf fonts-tlwg-typist
  fonts-tlwg-typist-ttf fonts-tlwg-typo fonts-tlwg-typo-ttf fonts-tlwg-umpush
  fonts-tlwg-umpush-ttf fonts-tlwg-waree fonts-tlwg-waree-ttf
The following NEW packages will be installed:
  fonts-thai-tlwg fonts-tlwg-garuda fonts-tlwg-garuda-ttf fonts-tlwg-kinnari
  fonts-tlwg-kinnari-ttf fonts-tlwg-laksaman fonts-tlwg-laksaman-ttf
  fonts-tlwg-loma fonts-tlwg-loma-ttf fonts-tlwg-mono fonts-tlwg-mono-ttf
  fonts-tlwg-norasi fonts-tlwg-norasi-ttf fo

---

## ส่วนที่ 1 — เตรียม Class และฟังก์ชัน

มีทั้งหมด 3 คลาส คือ

1. **Class Member** (สมาชิก) — รหัสสมาชิก, ชื่อสมาชิก, เลขบัตรประจำตัวประชาชน, เบอร์โทรศัพท์
2. **Class Account** (บัญชีออมทรัพย์) — เลขบัญชี, ยอดเงินคงเหลือ, เจ้าของบัญชี, ดอกเบี้ยต่อปี
3. **Class Transaction** (ธุรกรรม) — หมายเลขธุรกรรม, บัญชี, ประเภทธุรกรรม, จำนวนเงิน, บัญชีปลายทาง

In [3]:
class Member:
    """ข้อมูลสมาชิกธนาคารออมทรัพย์"""

    def __init__(
        self,
        member_id,
        customer_name,
        citizen_id=None,
        phone_number=None,
    ):
        self.member_id = member_id
        self.customer_name = customer_name
        self.citizen_id = citizen_id
        self.phone_number = phone_number

    # แสดงข้อมูลสมาชิก
    def get_info(self):
        return (
            f"ลูกค้า ID: {self.member_id} | ชื่อ: {self.customer_name} | "
            f"เลขบัตรประชาชน: {self.citizen_id} | เบอร์โทร: {self.phone_number}"
        )

    # อัปเดตข้อมูลส่วนตัว
    def update_phone(self, new_phone):
        self.phone_number = new_phone

In [4]:
class Account:
    """บัญชีออมทรัพย์"""

    def __init__(
        self,
        account_number,
        balance,
        owner,
        interest_rate=0.015,
    ):
        self.account_number = account_number
        self.balance = float(balance)
        self.owner = owner
        self.interest_rate = interest_rate

    def deposit(self, amount):
        """ฝากเงิน: balance = balance + amount"""
        self.balance += amount
        return "ฝากเงินสำเร็จ"

    def withdraw(self, amount):
        """ถอนเงิน: ตรวจสอบ balance >= amount"""
        if self.balance < amount:
            return f"ยอดเงินไม่พอ (มีอยู่ {self.balance:,.2f} บาท)"

        self.balance -= amount
        return "ถอนเงินสำเร็จ"

    def transfer(self, target_account, amount):
        """โอนเงิน: ตัดบัญชีต้นทาง และบวกเข้าบัญชีปลายทาง"""
        if self.balance < amount:
            return f"ยอดเงินไม่พอโอน (มีอยู่ {self.balance:,.2f} บาท)"

        self.balance -= amount
        target_account.balance += amount
        return "โอนเงินสำเร็จ"

    def apply_interest(self):
        """คำนวณดอกเบี้ยและบวกเข้ายอดคงเหลือ"""
        interest = self.balance * self.interest_rate
        self.balance += interest
        return interest

In [5]:
def generate_transaction_data(txn_id):
    """ฟังก์ชันสุ่มคิว 40 รายการต่อวัน (รับแค่ txn_id)"""
    base_date = datetime(2026, 8, 22)
    day_idx = 0
    total_items = 0

    while True:
        items_today = 40

        # เช็คว่า txn_id นี้ยังอยู่ในโควตาสะสมของวันนี้หรือไม่
        if txn_id <= total_items + items_today:
            queue_num = txn_id - total_items
            current_date = base_date + timedelta(days=day_idx)

            return {
                "วันที่": current_date.strftime("%d/%m/%Y"),
                "หมายเลขคิว": f"A-{queue_num:03d}",
                "queue_seq": queue_num - 1,
            }

        # ย้ายเข้ามาอยู่ใน while เพื่อขยับรอบวัน (แก้ Infinite Loop)
        total_items += items_today
        day_idx += 1

In [6]:
class Transaction:

    def __init__(
        self,
        txn_id,
        account,
        transaction_type,
        amount,
        target_account=None,
    ):
        self.txn_id = txn_id

        # 1. เรียกใช้สุ่มคิวตาม txn_id
        date_info = generate_transaction_data(txn_id)
        self.queue_number = date_info["หมายเลขคิว"]
        self.txn_date = date_info["วันที่"]

        # 2. คำนวณเวลาตามลำดับคิวในวันนั้น
        base_start_time = datetime.strptime("08:30:00", "%H:%M:%S")
        queue_seq = date_info["queue_seq"]

        minutes_added = (queue_seq * random.randint(8, 11)) + random.randint(
            0, 2
        )
        seconds_added = random.randint(0, 59)

        actual_time = base_start_time + timedelta(
            minutes=minutes_added, seconds=seconds_added
        )
        self.time = actual_time.strftime("%H:%M:%S")

        # 3. จัดเก็บข้อมูลจาก Account
        self.account = account
        self.account_number = account.account_number
        self.customer_name = account.owner.customer_name
        self.transaction_type = transaction_type
        self.amount = amount
        self.target_account = target_account

    def to_dict(self):
        # คำนวณดอกเบี้ยจาก Account
        interest = self.account.balance * self.account.interest_rate

        # แยกชื่อ - นามสกุล
        if isinstance(self.customer_name, (tuple, list)):
            fname, lname = self.customer_name[0], self.customer_name[1]
        else:
            parts = str(self.customer_name).split(" ", 1)
            fname = parts[0]
            lname = parts[1] if len(parts) > 1 else "-"

        # ดึงเลขบัญชีปลายทาง
        if self.target_account:
            target_acc_no = getattr(
                self.target_account, "account_number", str(self.target_account)
            )
        else:
            target_acc_no = "-"

        return {
            "ID รายการ": self.txn_id,
            "หมายเลขคิว": self.queue_number,
            "วันที่ทำรายการ": self.txn_date,
            "เวลาทำรายการ": self.time,
            "เลขบัญชี": self.account_number,
            "ชื่อ": fname,
            "นามสกุล": lname,
            "ประเภทรายการ": self.transaction_type,
            "จำนวนเงิน": self.amount,
            "บัญชีปลายทาง": target_acc_no,
            "ยอดหลังทำรายการ": round(self.account.balance, 2),
            "ดอกเบี้ยสิ้นปี (1.5%)": round(interest, 2),
            "ยอดรวมดอกเบี้ยสุทธิ": round(self.account.balance + interest, 2),
        }

#ส่วนที่ 2 — ฟังก์ชันช่วยงาน (Helper Function)

In [7]:
def generate_thai_name():
    """ฟังก์ชัน: สุ่มชื่อและนามสกุลลูกค้าแยกกัน"""
    name = fake.name()
    first_name, last_name = name.split(" ", 1)
    return f"{first_name} {last_name}"


def random_amount(min_val=100.0, max_val=2000.0):
    """ฟังก์ชัน: สุ่มยอดเงิน -> คืนค่าเป็น float"""
    return round(random.uniform(min_val, max_val), 2)


def format_currency(amount, symbol="บาท"):
    """ฟังก์ชัน: จัดรูปแบบตัวเลขเป็นสตรีงราคา -> คืนค่าเป็น string"""
    return f"{amount:,.2f} {symbol}"

##ส่วนที่ 2.1 — ทดสอบสุ่มชื่อ

In [8]:
# เรียก generate_thai_name() 3 ครั้ง -> ทุกครั้งได้ชื่อสุ่มไม่ซ้ำแบบ
for _ in range(3):
    print("ชื่อที่สุ่มได้:", generate_thai_name())

ชื่อที่สุ่มได้: ประยุทธ์ เยาวธนโชค
ชื่อที่สุ่มได้: ปิยนุช ปานสุวรรณ
ชื่อที่สุ่มได้: โชติวุฒิ นพตระกูล


#ส่วนที่ 2.2 — ทดสอบสุ่มยอดเงิน

In [9]:
print("\n# เรียก random_amount() 3 ครั้ง")
for _ in range(3):
    print("ยอดเงินสุ่มได้:", format_currency(random_amount()))


# เรียก random_amount() 3 ครั้ง
ยอดเงินสุ่มได้: 355.29 บาท
ยอดเงินสุ่มได้: 1,710.12 บาท
ยอดเงินสุ่มได้: 1,551.17 บาท


##ส่วนที่ 2.3 — ทดสอบอัพเดทเบอร์โทรศัพท์

In [10]:
# 1. ทดสอบสร้าง Instance ของ Member
demo_member = Member(member_id="M001", customer_name="สมชาย ใจดี", citizen_id="1234567890123", phone_number="0812345678")
print("=== [1] ทดสอบ Method ของ Class Meส่วนที่ 2.3 — ทดสอบอัพเดทเบอร์โทรศัพท์$0mber ===")
print(demo_member.get_info())

# 🔹 เพิ่มการทดสอบอัปเดตเบอร์ตรงนี้ 🔹
demo_member.update_phone("0896794152")
print("หลังอัปเดตเบอร์:", demo_member.get_info())

# 2. ทดสอบสร้าง Instance ของ Account โดยผูกกับ demo_member
demo_account = Account(account_number="100-1-00001-0", balance=1000.0, owner=demo_member)
print("\n=== [2] ทดสอบ Method ของ Class Account ===")
print(f"ยอดเงินเริ่มต้น: {demo_account.balance:,.2f} บาท")
print(f"ผลการฝากเงิน 500 บาท: {demo_account.deposit(500)}")
print(f"ยอดเงินหลังฝาก: {demo_account.balance:,.2f} บาท")
print(f"ผลการถอนเงิน 2,000 บาท: {demo_account.withdraw(2000)}")

=== [1] ทดสอบ Method ของ Class Meส่วนที่ 2.3 — ทดสอบอัพเดทเบอร์โทรศัพท์$0mber ===
ลูกค้า ID: M001 | ชื่อ: สมชาย ใจดี | เลขบัตรประชาชน: 1234567890123 | เบอร์โทร: 0812345678
หลังอัปเดตเบอร์: ลูกค้า ID: M001 | ชื่อ: สมชาย ใจดี | เลขบัตรประชาชน: 1234567890123 | เบอร์โทร: 0896794152

=== [2] ทดสอบ Method ของ Class Account ===
ยอดเงินเริ่มต้น: 1,000.00 บาท
ผลการฝากเงิน 500 บาท: ฝากเงินสำเร็จ
ยอดเงินหลังฝาก: 1,500.00 บาท
ผลการถอนเงิน 2,000 บาท: ยอดเงินไม่พอ (มีอยู่ 1,500.00 บาท)


## ส่วนที่ 3 — ฟังก์ชันอธิบายขั้นตอนคำนวณราคา

In [11]:
def explain_transaction_calculation(transaction):
    """ฟังก์ชันคำนวณเงิน ฝาก/ถอน/โอน และเรียกใช้ Method ของ Account"""

    account = transaction.account
    amount = float(transaction.amount)
    txn_type = transaction.transaction_type

    print(f"หมายเลขคิว = '{transaction.queue_number}'")
    print(f"หมายเลขบัญชี = '{account.account_number}'")
    print(f"ชื่อลูกค้า = '{transaction.customer_name}'")
    print(f"ประเภทรายการ = '{txn_type}'")
    print(f"ยอดเงินก่อนทำรายการ = {format_currency(account.balance)}")

    # เรียกใช้ Method ภายใน Class Account
    if txn_type == "ฝากเงิน":
      status_msg = account.deposit(amount)

    elif txn_type == "ถอนเงิน":
        status_msg = account.withdraw(amount)
        # ถ้าเงินไม่พอ ให้หยุดประมวลผลทันที
        if "ยอดเงินไม่พอ" in status_msg:
          print(f"จำนวนเงินทำรายการ = {format_currency(amount)}")
          print(f"ยอดเงินคงเหลือหลังทำรายการ = {format_currency(account.balance)}")
          print(f"สถานะรายการ = '{status_msg}'")
          print("❌ ทำรายการไม่สำเร็จ!")
          return False

    elif txn_type == "โอนเงิน":
        if transaction.target_account:
            print(f"บัญชีปลายทาง = '{transaction.target_account}'")

        dummy_member = Member(0, "บัญชีปลายทาง")
        dummy_target = Account("987-6-00000-0", balance=0.0, owner=dummy_member)
        status_msg = account.transfer(dummy_target, amount)
        # ถ้าเงินไม่พอโอน ให้หยุดประมวลผลทันที
        if "ยอดเงินไม่พอ" in status_msg:
            print(f"จำนวนเงินทำรายการ = {format_currency(amount)}")
            print(f"ยอดเงินคงเหลือหลังทำรายการ = {format_currency(account.balance)}")
            print(f"สถานะรายการ = '{status_msg}'")
            print("❌ ทำรายการไม่สำเร็จ!")
            return False

    # คำนวณดอกเบี้ย (จะทำเฉพาะรายการที่สำเร็จเท่านั้น)
    interest_val = account.apply_interest()

    print(f"จำนวนเงินทำรายการ = {format_currency(amount)}")
    print(f"ยอดเงินคงเหลือหลังทำรายการ = {format_currency(account.balance)}")
    print(f"สถานะรายการ = '{status_msg}'")
    print(f"ดอกเบี้ยที่ได้รับเมื่อสิ้นปี (1.5%) = {format_currency(interest_val)}")

    return True

In [12]:
import random

random.seed(1)
fake.seed_instance(1)  # ใช้ fake.seed_instance(1) เพื่อล็อกค่าตัวแปร fake โดยตรง

transactions = []

# Loop สุ่มข้อมูล 300 รายการ
for i in range(1, 301):
    name = generate_thai_name()
    amount = random_amount()
    service = random.choice(["ฝากเงิน", "ถอนเงิน", "โอนเงิน"])

    member = Member(member_id=1 + i, customer_name=name)

    initial_balance = round(random.uniform(100, 3000), 2)

    # สุ่มเลขบัญชีลูกค้า
    acc_p1 = random.randint(100, 999)
    acc_p2 = random.randint(1, 9)
    acc_p3 = random.randint(10000, 99999)
    random_account_no = f"{acc_p1}-{acc_p2}-{acc_p3:05d}-0"

    account = Account(account_number=random_account_no, balance=initial_balance, owner=member)
    target_acc = f"987-6-{random.randint(10000, 99999)}-0" if service == "โอนเงิน" else None
    # คำนวณยอดเงินผ่าน Method ของ Account โดยตรง
    if service == "ฝากเงิน":
        account.deposit(amount)
    elif service == "ถอนเงิน":
        account.withdraw(amount)
    elif service == "โอนเงิน":
        dummy_mem = Member(0, "ปลายทาง")
        dummy_acc = Account("987-6-00000-0", balance=0.0, owner=dummy_mem)
        account.transfer(dummy_acc, amount)

    # คำนวณเลขคิวให้รีเซ็ตทุกๆ 40 คิว
    daily_queue = ((i - 1) % 40) + 1
    queue_no = f"A-{daily_queue:03d}"

    # ประมวลผลดอกเบี้ย
    account.apply_interest()

    transaction = Transaction(
        txn_id=i,
        account=account,
        transaction_type=service,
        amount=amount,
        target_account=target_acc
    )

    transactions.append(transaction)

# 3. แสดงตัวอย่างรายการแรก (คิว A-001)
print("\n--- [ตัวอย่างการแสดงผลรายการแรก (คิว A-001)] ---")
explain_transaction_calculation(transactions[0])


--- [ตัวอย่างการแสดงผลรายการแรก (คิว A-001)] ---
หมายเลขคิว = 'A-001'
หมายเลขบัญชี = '607-8-71898-0'
ชื่อลูกค้า = 'ชิดชนก เยาวธนโชค'
ประเภทรายการ = 'ฝากเงิน'
ยอดเงินก่อนทำรายการ = 1,212.91 บาท
จำนวนเงินทำรายการ = 355.29 บาท
ยอดเงินคงเหลือหลังทำรายการ = 1,591.73 บาท
สถานะรายการ = 'ฝากเงินสำเร็จ'
ดอกเบี้ยที่ได้รับเมื่อสิ้นปี (1.5%) = 23.52 บาท


True

> 💡 **หมายเหตุ:** เป็นการล็อกค่าของการสุ่ม (Random Seed) ไว้ เพื่อให้ทุกครั้งที่กดรันโปรแกรม ระบบจะสุ่มได้ตัวเลขและข้อมูลชุดเดิมเสมอ ทำให้ง่ายต่อการทดสอบและตรวจสอบความถูกต้องของระบบ